# FAOD — Frequency-Adaptive Object Detection
## End-to-End Training Notebook

This notebook walks through the complete FAOD training pipeline on DSEC-Det:
1. **Setup** — imports, paths, config
2. **Dataset** — loading DSEC event + frame data
3. **Dataloader** — batching sequences
4. **Model** — building the FAOD network (RNN backbone + YoloX head)
5. **Training & Validation** — running the training loop

### Architecture Overview
```
Events (stacked histogram) ──┐
                              ├── Cross-CBAM Fusion ── LSTM RNN ── FPN ── YoloX Head ── Detections
RGB Frames ──────────────────┘
```
- **Backbone**: DarkNet with LSTM memory (recurrent, processes sequences of 5 timesteps)
- **FPN**: Path Aggregation FPN for multi-scale features
- **Head**: YoloX detection head (8 classes)

## Cell 1: Environment Setup

We set up paths and environment variables needed before importing anything.
- `HDF5_PLUGIN_PATH` is required to read DSEC `.h5` files which use blosc compression
- We add the FAOD root to sys.path so all imports work from this notebook

In [ ]:
import os
import sys
from pathlib import Path

# ── Point to FAOD root (one level up from this notebook) ──────────────────────
FAOD_ROOT = Path("__file__").resolve().parent.parent
# If running interactively, set manually:
FAOD_ROOT = Path("/home/karthik/Desktop/spiking_neural_networks/FAOD-master")

# Add FAOD root to Python path so all internal imports resolve
if str(FAOD_ROOT) not in sys.path:
    sys.path.insert(0, str(FAOD_ROOT))

# Change working directory to FAOD root (Hydra config resolution requires this)
os.chdir(FAOD_ROOT)

# ── HDF5 plugin path for blosc-compressed DSEC files ──────────────────────────
import importlib.util
_spec = importlib.util.find_spec('hdf5plugin')
if _spec:
    os.environ['HDF5_PLUGIN_PATH'] = str(Path(_spec.origin).parent / 'plugins')
import hdf5plugin  # must import after setting env var

# ── Disable wandb (no API key needed) ─────────────────────────────────────────
os.environ['WANDB_MODE'] = 'disabled'

print(f"FAOD_ROOT: {FAOD_ROOT}")
print(f"HDF5_PLUGIN_PATH: {os.environ.get('HDF5_PLUGIN_PATH', 'not set')}")
print(f"WANDB_MODE: {os.environ['WANDB_MODE']}")

## Cell 2: Configuration

FAOD uses **Hydra** for config management. We compose the config manually here
(same as what `train.py` does) so we can inspect and modify it in the notebook.

Key config sections:
- `dataset` — path, event representation, sequence length
- `model` — backbone type, embed_dim, memory type
- `training` — learning rate, max steps, LR scheduler
- `hardware` — GPU, num_workers, batch size

In [ ]:
from omegaconf import OmegaConf, DictConfig
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

# Clear any previous Hydra state
GlobalHydra.instance().clear()

# Initialize Hydra with FAOD's config directory
with initialize_config_dir(config_dir=str(FAOD_ROOT / 'config'), version_base='1.2'):
    config = compose(
        config_name='train',
        overrides=[
            # Dataset: use our small DSEC subset
            'dataset=dsec',
            'dataset.path=data/dsec_small_h5/freq_1_1',
            # Experiment: tiny model (smaller embed_dim for fast training)
            '+experiment/dsec=tiny.yaml',
            # Disable deformable alignment (requires mmcv CUDA match)
            'model.backbone.enable_align=False',
            # Batch sizes
            'batch_size.train=2',
            'batch_size.eval=1',
            # Workers (1 because we only have 1 sequence)
            'hardware.num_workers.train=1',
            'hardware.num_workers.eval=1',
            # Training length
            'training.max_steps=2000',
            'training.lr_scheduler.total_steps=2000',
            # Validate every 500 steps
            'validation.val_check_interval=500',
            # Stream sampling (required for single-sequence training)
            'dataset.train.sampling=stream',
        ]
    )

# Apply dynamic modifications (sets num_classes, input resolution, etc.)
from config.modifier import dynamically_modify_train_config
dynamically_modify_train_config(config)

print("=== Key Config Values ===")
print(f"Dataset path  : {config.dataset.path}")
print(f"Event repr    : {config.dataset.ev_repr_name}")
print(f"Sequence len  : {config.dataset.sequence_length}")
print(f"Resolution    : {config.dataset.resolution_hw}")
print(f"Backbone type : {config.model.backbone.backbone_type}")
print(f"Memory type   : {config.model.backbone.memory_type}")
print(f"Embed dim     : {config.model.backbone.embed_dim}")
print(f"Num classes   : {config.model.head.num_classes}")
print(f"Max steps     : {config.training.max_steps}")
print(f"Learning rate : {config.training.learning_rate}")
print(f"Batch train   : {config.batch_size.train}")
print(f"Batch eval    : {config.batch_size.eval}")

## Cell 3: Dataset Loading

FAOD uses a **streaming dataset** for training — it iterates through sequences
in order (important for the RNN to maintain temporal state).

Each dataset item contains:
- `DataType.EV_REPR` — stacked histogram of events, shape `(nbins*2, H, W)` = `(20, H, W)`
- `DataType.IMAGE` — RGB frame, shape `(3, H, W)`
- `DataType.OBJLABELS_SEQ` — bounding box labels for each timestep
- `DataType.IS_FIRST_SAMPLE` — flag to reset RNN state at sequence start

The preprocessing pipeline (`frame_construction/main_dsec.py`) already converted
raw DSEC events into these stacked histogram `.h5` files.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from data.utils.types import DatasetMode, DatasetSamplingMode, DataType

# ── Build the streaming train dataset ─────────────────────────────────────────
# Streaming dataset: iterates samples in temporal order (required for RNN)
# Each "sample" is a window of sequence_length=5 consecutive timesteps
from data.ev_img_dataloader.dataset_streaming import build_streaming_dataset

print("Building streaming train dataset...")
train_dataset_info = build_streaming_dataset(
    dataset_mode=DatasetMode.TRAIN,          # Train split
    dataset_config=config.dataset,           # Config with path, ev_repr_name, etc.
    batch_size=config.batch_size.train,      # 2
    num_workers=config.hardware.num_workers.train  # 1
)

# ── Build the streaming val dataset ───────────────────────────────────────────
print("Building streaming val dataset...")
val_dataset_info = build_streaming_dataset(
    dataset_mode=DatasetMode.VALIDATION,
    dataset_config=config.dataset,
    batch_size=config.batch_size.eval,
    num_workers=config.hardware.num_workers.eval
)

print(f"\nTrain dataset type : {type(train_dataset_info)}")
print(f"Val dataset type   : {type(val_dataset_info)}")

# Peek at a single raw sample from the train dataset
# (train_dataset_info is a torchdata DataPipeline for streaming)
print("\nDataset ready. Data types available per sample:")
for dtype in DataType:
    print(f"  {dtype.name}")

## Cell 4: DataModule and DataLoaders

The `DataModule` wraps both train and val datasets into PyTorch Lightning's
`LightningDataModule`. It handles:
- Creating streaming pipelines for temporal data
- Custom collation (grouping variable-length label sequences into batches)
- Proper worker setup for multi-process loading

In [ ]:
from modules.utils.fetch import fetch_data_module

# ── Create DataModule ──────────────────────────────────────────────────────────
# This is a PyTorch Lightning LightningDataModule that wraps train/val/test
# datasets and exposes train_dataloader(), val_dataloader(), test_dataloader()
print("Creating DataModule...")
data_module = fetch_data_module(config=config)

# Set up the datasets (calls setup() internally)
data_module.setup(stage='fit')

# ── Inspect a batch ───────────────────────────────────────────────────────────
# Get one batch from the val dataloader to inspect shapes
val_loader = data_module.val_dataloader()

print("\nInspecting one validation batch...")
for batch in val_loader:
    print("\nBatch contents:")
    for key, value in batch.items():
        if isinstance(value, torch.Tensor):
            print(f"  {key.name:25s}: Tensor {tuple(value.shape)} dtype={value.dtype}")
        elif isinstance(value, list):
            print(f"  {key.name:25s}: List[{type(value[0]).__name__}] len={len(value)}")
        else:
            print(f"  {key.name:25s}: {type(value).__name__}")
    break  # Only look at first batch

print(f"\nEvent repr shape  : (batch, nbins*2, H, W) where nbins=10 polarities=2")
print(f"Image shape       : (batch, 3, H, W) — RGB frame")

## Cell 5: Visualize a Sample

Let's visualize one sample from the dataset — events and the paired RGB frame.

The event representation is a **stacked histogram**:
- 20 channels total: 10 time bins × 2 polarities (positive/negative events)
- Brighter pixels = more events in that time bin at that location

In [ ]:
import matplotlib.patches as patches

# Get a sample batch
for batch in val_loader:
    ev_repr = batch[DataType.EV_REPR]   # (B, 20, H, W)
    image   = batch[DataType.IMAGE]     # (B, 3, H, W)
    labels  = batch.get(DataType.OBJLABELS_SEQ, None)
    break

# ── Plot events and image side by side ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('DSEC-Det Sample — Events + RGB Frame', fontsize=14)

# Event representation: sum all channels to get an "event density" image
ev_sum = ev_repr[0].sum(dim=0).cpu().numpy()  # (H, W)
axes[0].imshow(ev_sum, cmap='hot')
axes[0].set_title('Events (sum of all 20 channels)\nHot = more events')
axes[0].axis('off')

# Positive events only (channels 0-9)
ev_pos = ev_repr[0, :10].sum(dim=0).cpu().numpy()
axes[1].imshow(ev_pos, cmap='Blues')
axes[1].set_title('Positive Events (brightness = motion)')
axes[1].axis('off')

# RGB frame
img_np = image[0].permute(1, 2, 0).cpu().numpy()  # (H, W, 3)
img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
axes[2].imshow(img_np)
axes[2].set_title('RGB Frame')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('notebooks/sample_visualization.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Event repr shape : {ev_repr.shape}")
print(f"  20 channels = 10 time bins × 2 polarities")
print(f"Image shape      : {image.shape}")

## Cell 6: Model Architecture

The FAOD model has 3 main components:

1. **Recurrent Backbone** (RNNDetector)
   - Takes events + RGB frames as input
   - Fuses them using Cross-CBAM attention
   - Maintains temporal state with LSTM cells at each scale level
   - DarkNet-style stem + 4 stages with increasing channel depth

2. **FPN** (Path Aggregation FPN)
   - Takes multi-scale features from stages 2, 3, 4 of backbone
   - Creates a feature pyramid for detecting objects at different scales

3. **YoloX Detection Head**
   - Predicts bounding boxes + class scores at each scale
   - 8 classes: pedestrian, rider, car, bus, truck, bicycle, motorcycle, train

In [ ]:
from modules.utils.fetch import fetch_model_module

# ── Build the model ────────────────────────────────────────────────────────────
# fetch_model_module reads config.model.backbone.type ('fusion') and
# returns a PyTorch Lightning Module wrapping the YoloXDetector
print("Building FAOD model...")
module = fetch_model_module(config=config)

print("\n=== Model Summary ===")
print(f"Backbone type : {config.model.backbone.backbone_type} (DarkNet)")
print(f"Fusion type   : {config.model.backbone.fusion_type} (Cross-CBAM)")
print(f"Memory type   : {config.model.backbone.memory_type} (LSTM)")
print(f"Embed dim     : {config.model.backbone.embed_dim} (32 for tiny)")
print(f"Num classes   : {config.model.head.num_classes}")

# Count parameters
total_params = sum(p.numel() for p in module.parameters())
trainable_params = sum(p.numel() for p in module.parameters() if p.requires_grad)
print(f"\nTotal parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size (MB)     : {total_params * 4 / 1e6:.1f}")

# Show sub-module breakdown
print("\n=== Sub-modules ===")
for name, child in module.mdl.named_children():
    params = sum(p.numel() for p in child.parameters())
    print(f"  {name:20s}: {params:>10,} params")

## Cell 7: Model Forward Pass (Inference Test)

Let's verify the model can process one batch before training.
This also demonstrates the RNN state management:
- LSTM states are initialized to `None` at the start of each sequence
- They are carried forward across timesteps within a sequence
- `IS_FIRST_SAMPLE=True` tells the model to reset states

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Move model to GPU
module = module.to(device)
module.eval()  # Inference mode

# Get a batch and move to device
for batch in val_loader:
    batch_gpu = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                 for k, v in batch.items()}
    break

# ── Run one forward pass ───────────────────────────────────────────────────────
# The model internally manages LSTM states.
# We reset them manually before starting a new sequence.
with torch.no_grad():
    # Reset RNN states (done automatically on IS_FIRST_SAMPLE=True)
    module.reset_states()
    
    # Run validation step (includes NMS post-processing)
    ev = batch_gpu[DataType.EV_REPR]    # (B, 20, H, W)
    img = batch_gpu[DataType.IMAGE]     # (B, 3, H, W)
    
    # Forward through backbone + FPN + head
    output = module.mdl(ev_tensor=ev, img_tensor=img)

print("Forward pass succeeded!")
print(f"\nInput event tensor : {ev.shape}")
print(f"Input image tensor : {img.shape}")
if isinstance(output, (list, tuple)):
    print(f"Output (raw)       : {[o.shape for o in output if isinstance(o, torch.Tensor)]}")
else:
    print(f"Output type        : {type(output)}")

## Cell 8: Training Setup

We use **PyTorch Lightning's Trainer** which handles:
- GPU management
- Gradient clipping
- Checkpoint saving
- Validation loop
- Learning rate scheduling

The optimizer is **AdamW** with **OneCycleLR** scheduler:
- Warmup for first 0.5% of steps
- Peak LR = 0.00015
- Cosine annealing to near zero

The loss is computed inside `YoloX` head — it combines:
- **IoU loss** for box regression
- **BCE loss** for objectness
- **BCE loss** for class prediction

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import LearningRateMonitor, ModelSummary
from callbacks.custom import get_ckpt_callback, get_viz_callback
from callbacks.gradflow import GradFlowLogCallback
from loggers.wandb_logger import WandbLogger
import wandb

# ── Logger (disabled — no wandb account needed) ────────────────────────────────
# WandbLogger with WANDB_MODE=disabled creates a local-only run
run_id = wandb.util.generate_id()
logger = WandbLogger(
    name=config.wandb.name,
    project=config.wandb.project_name,
    group=config.wandb.group_name,
    wandb_id=run_id,
    log_model=False,    # Don't upload checkpoints to wandb (disabled mode)
    config_args=OmegaConf.to_container(config, resolve=True),
)
print(f"Run ID: {run_id}")

# ── Callbacks ─────────────────────────────────────────────────────────────────
callbacks = []

# 1. Checkpoint callback — saves best model by val/AP
ckpt_callback = get_ckpt_callback(config)
callbacks.append(ckpt_callback)
print(f"Checkpoints saved to: dummy/{run_id}/checkpoints/")

# 2. Gradient flow logging (logs gradient norms every N steps)
callbacks.append(GradFlowLogCallback(
    config.logging.train.log_model_every_n_steps
))

# 3. Learning rate monitor
if config.training.lr_scheduler.use:
    callbacks.append(LearningRateMonitor(logging_interval='step'))

# 4. Model summary
callbacks.append(ModelSummary(max_depth=2))

print(f"\nCallbacks: {[type(c).__name__ for c in callbacks]}")

# ── Trainer ───────────────────────────────────────────────────────────────────
trainer = pl.Trainer(
    accelerator='gpu',
    devices=[0],                      # Use GPU 0 (RTX 5090)
    callbacks=callbacks,
    logger=logger,
    
    # Training length
    max_epochs=config.training.max_epochs,        # 10000 (hit max_steps first)
    max_steps=config.training.max_steps,          # 2000 steps total
    
    # Validation frequency
    val_check_interval=config.validation.val_check_interval,    # every 500 steps
    check_val_every_n_epoch=config.validation.check_val_every_n_epoch,
    
    # Gradient clipping (prevents exploding gradients in RNN)
    gradient_clip_val=config.training.gradient_clip_val,        # 1.0
    gradient_clip_algorithm='value',
    
    # Precision (32-bit float)
    precision=config.training.precision,
    
    # Logging
    log_every_n_steps=config.logging.train.log_every_n_steps,   # every 500 steps
    
    # Batch limits
    limit_train_batches=config.training.limit_train_batches,    # 1.0 = all
    limit_val_batches=config.validation.limit_val_batches,      # 1.0 = all
    
    benchmark=config.reproduce.benchmark,
    deterministic=config.reproduce.deterministic_flag,
)

print("\nTrainer ready!")
print(f"  Max steps       : {config.training.max_steps}")
print(f"  Val every       : {config.validation.val_check_interval} steps")
print(f"  Gradient clip   : {config.training.gradient_clip_val}")
print(f"  Precision       : {config.training.precision}-bit")

## Cell 9: Rebuild Model (fresh weights for training)

We rebuild the model fresh before training (the earlier forward pass moved
it to GPU and changed its state). This ensures training starts from scratch.

In [ ]:
from omegaconf import OmegaConf

# Rebuild model with fresh random weights
module = fetch_model_module(config=config)

# Rebuild data module
data_module = fetch_data_module(config=config)

print("Model and DataModule ready for training.")
print(f"\nWhat will happen during training:")
print(f"  Step 1-500  : Train, then validate → save checkpoint")
print(f"  Step 501-1000: Train, then validate → update checkpoint if better")
print(f"  Step 1001-1500: Train, then validate")
print(f"  Step 1501-2000: Train, then final validate → done")
print(f"\nExpected loss range: starts ~70-80, drops to ~5-15 after 2000 steps")
print(f"val/AP will be low (0.00) on this tiny single-sequence subset — expected")

## Cell 10: Training

Run `trainer.fit()` to start training. This will:
1. Run a sanity validation check (2 batches)
2. Start training loop — each step processes `sequence_length=5` timesteps
3. Every 500 steps: run full validation and save checkpoint
4. Stop at 2000 steps

**Approximate time**: ~5 minutes on RTX 5090

In [ ]:
import time

print("Starting training...")
print("=" * 60)
t0 = time.time()

# ── START TRAINING ─────────────────────────────────────────────────────────────
# trainer.fit() runs the full training loop:
#   - training_step()  : forward pass + loss + backward + optimizer step
#   - validation_step(): forward pass only (no gradient), compute AP
#   - on_validation_epoch_end(): aggregate AP across all val sequences
trainer.fit(
    model=module,
    datamodule=data_module,
    ckpt_path=None  # None = train from scratch
)

elapsed = time.time() - t0
print("=" * 60)
print(f"Training complete! Total time: {elapsed:.0f}s ({elapsed/60:.1f} min)")

## Cell 11: Training Results

Let's inspect the saved checkpoints and training metrics.

In [ ]:
import glob

# ── Find saved checkpoints ─────────────────────────────────────────────────────
ckpt_dir = FAOD_ROOT / 'dummy' / run_id / 'checkpoints'
checkpoints = sorted(ckpt_dir.glob('*.ckpt'))

print(f"Checkpoints saved in: {ckpt_dir}")
print(f"\nSaved checkpoints:")
for ckpt in checkpoints:
    size_mb = ckpt.stat().st_size / 1e6
    print(f"  {ckpt.name}  ({size_mb:.1f} MB)")

# ── Load best checkpoint info ──────────────────────────────────────────────────
best_ckpt = ckpt_callback.best_model_path
best_score = ckpt_callback.best_model_score
print(f"\nBest checkpoint : {Path(best_ckpt).name}")
print(f"Best val/AP     : {best_score}")
print(f"\nNote: val/AP=0.00 is expected on a single tiny sequence.")
print(f"For meaningful AP, train on the full DSEC-Det dataset (60 sequences).")

## Cell 12: Validation / Inference on Test Set

Run the trained model on the test set to get final mAP scores.
We load the best checkpoint and run `trainer.test()`.

In [ ]:
# ── Load best checkpoint ───────────────────────────────────────────────────────
best_ckpt_path = ckpt_callback.best_model_path
print(f"Loading checkpoint: {Path(best_ckpt_path).name}")

# Load weights into a fresh module instance
from modules.detection_fusion import Module as FusionModule
trained_module = FusionModule.load_from_checkpoint(
    best_ckpt_path,
    full_config=config
)
trained_module.eval()

# ── Create a test-mode data module ────────────────────────────────────────────
# Override config to use test set
from copy import deepcopy
test_config = deepcopy(config)
OmegaConf.update(test_config, 'validation.limit_val_batches', 1.0)

test_data_module = fetch_data_module(config=test_config)

# ── Run test ───────────────────────────────────────────────────────────────────
test_trainer = pl.Trainer(
    accelerator='gpu',
    devices=[0],
    logger=False,       # No logging for test run
    enable_progress_bar=True,
)

print("\nRunning test evaluation...")
test_results = test_trainer.test(
    model=trained_module,
    datamodule=test_data_module,
    ckpt_path=best_ckpt_path,
    verbose=True
)

print("\nTest Results:")
for key, val in test_results[0].items():
    print(f"  {key}: {val:.4f}")

## Cell 13: Visualize Predictions

Run the trained model on a few frames and visualize the detections.

DSEC-Det class labels:
- 0: pedestrian, 1: rider, 2: car, 3: bus
- 4: truck, 5: bicycle, 6: motorcycle, 7: train

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

CLASS_NAMES = ['pedestrian', 'rider', 'car', 'bus', 'truck', 'bicycle', 'motorcycle', 'train']
COLORS = plt.cm.Set1(np.linspace(0, 1, len(CLASS_NAMES)))

trained_module = trained_module.to(device)
trained_module.eval()
trained_module.reset_states()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('FAOD Predictions on DSEC-Det Test Sequence', fontsize=14)

axes_flat = axes.flatten()
frame_idx = 0

with torch.no_grad():
    for batch in val_loader:
        if frame_idx >= 6:
            break
        
        # Reset RNN state at start of new sequence
        is_first = batch.get(DataType.IS_FIRST_SAMPLE, [False])
        if any(is_first):
            trained_module.reset_states()
        
        ev  = batch[DataType.EV_REPR].to(device)
        img = batch[DataType.IMAGE].to(device)
        
        # Forward pass — returns raw predictions
        predictions = trained_module.mdl(ev_tensor=ev, img_tensor=img)
        
        # Post-process: apply NMS to get final boxes
        from models.detection.yolox_extension.utils import postprocess
        det_output = postprocess(
            predictions,
            num_classes=config.model.head.num_classes,
            conf_thre=config.model.postprocess.confidence_threshold,
            nms_thre=config.model.postprocess.nms_threshold,
        )
        
        # ── Visualize ────────────────────────────────────────────────────
        ax = axes_flat[frame_idx]
        img_np = img[0].permute(1, 2, 0).cpu().numpy()
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
        ax.imshow(img_np)
        ax.set_title(f'Frame {frame_idx}')
        ax.axis('off')
        
        # Draw predicted boxes
        if det_output[0] is not None:
            dets = det_output[0].cpu().numpy()  # (N, 7): x1,y1,x2,y2,score,cls_score,cls
            for det in dets:
                x1, y1, x2, y2, conf, cls_conf, cls_id = det
                cls_id = int(cls_id)
                color = COLORS[cls_id % len(COLORS)]
                rect = patches.Rectangle(
                    (x1, y1), x2 - x1, y2 - y1,
                    linewidth=2, edgecolor=color, facecolor='none'
                )
                ax.add_patch(rect)
                ax.text(x1, y1 - 2, f'{CLASS_NAMES[cls_id]} {conf:.2f}',
                        color=color, fontsize=7, fontweight='bold')
        
        frame_idx += 1

plt.tight_layout()
plt.savefig('notebooks/predictions.png', dpi=100, bbox_inches='tight')
plt.show()
print("Predictions saved to notebooks/predictions.png")

## Summary

| Step | What was done |
|------|---------------|
| Setup | Set HDF5 plugin path, disabled wandb, added FAOD to sys.path |
| Config | Composed Hydra config: tiny model, DSEC dataset, 2000 steps |
| Dataset | Built streaming DSEC dataset (events + frames + labels) |
| Dataloader | Created LightningDataModule with custom collation |
| Model | Built FAOD: DarkNet backbone + Cross-CBAM + LSTM + FPN + YoloX head |
| Training | Ran 2000 steps with AdamW + OneCycleLR, val every 500 steps |
| Results | Checkpoint saved, visualized predictions |

### To train on the full DSEC-Det dataset:
1. Download all 60 sequences
2. Run `prepare_dsec_small.py` (or modify paths in the script)
3. Run `frame_construction/main_dsec.py`
4. Change `max_steps=400000` and `dataset.path` to full dataset
5. Use `+experiment/dsec=base.yaml` instead of `tiny.yaml`